In [12]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "ma_walk_forward"
)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_DIR),
    )

pd.set_option(
    "display.max_columns",
    100,
)

pd.set_option(
    "display.float_format",
    lambda value: f"{value:.4f}",
)

plt.rcParams[
    "axes.unicode_minus"
] = False


%load_ext autoreload
%autoreload 2
from parameter_sensitivity import (
    build_ma_parameter_grid,
)

from walk_forward import (
    generate_walk_forward_windows,
    run_ma_walk_forward,
    run_fixed_parameter_validation,
    plot_walk_forward_metric,
    plot_selected_parameters,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
stock_list = [
    "000001",
    "000002",
    "300750",
    "600036",
    "600519",
]

fast_windows = [
    5,
    10,
    15,
    20,
    25,
    30,
    35,
    40,
]

slow_windows = [
    20,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    110,
    120,
]

parameter_grid = build_ma_parameter_grid(
    fast_windows=fast_windows,
    slow_windows=slow_windows,
    min_gap=10,
)

print(
    "参数组合数量：",
    len(parameter_grid),
)

参数组合数量： 76


In [ ]:
test_windows = (
    generate_walk_forward_windows(
        start_date="2015-01-01",
        end_date="2024-12-31",
        train_months=36,
        test_months=12,
        step_months=12,
        window_type="rolling",
        include_partial_test=False,
    )
)

display(test_windows)

,fold_id,window_type,train_start,train_end,test_start,test_end,train_months,test_months,step_months,is_partial_test
0,1,rolling,2015-01-01,2016-12-31,2017-01-01,2017-12-31,24,12,12,False
1,2,rolling,2016-01-01,2017-12-31,2018-01-01,2018-12-31,24,12,12,False
2,3,rolling,2017-01-01,2018-12-31,2019-01-01,2019-12-31,24,12,12,False
3,4,rolling,2018-01-01,2019-12-31,2020-01-01,2020-12-31,24,12,12,False
4,5,rolling,2019-01-01,2020-12-31,2021-01-01,2021-12-31,24,12,12,False
5,6,rolling,2020-01-01,2021-12-31,2022-01-01,2022-12-31,24,12,12,False
6,7,rolling,2021-01-01,2022-12-31,2023-01-01,2023-12-31,24,12,12,False
7,8,rolling,2022-01-01,2023-12-31,2024-01-01,2024-12-31,24,12,12,False


## 冒烟测试

In [15]:
smoke_grid = [
    (5, 30),
    (10, 40),
    (15, 50),
    (20, 60),
]

smoke_result = run_ma_walk_forward(
    stock_list=[
        "000001",
        "600036",
    ],
    parameter_grid=smoke_grid,
    train_months=36,
    test_months=12,
    step_months=12,
    window_type="rolling",
    start_date=None,
    end_date="2024-12-31",
    commission_rate=0.0003,
    slippage_rate=0.0002,
    robustness_metric=(
        "avg_strategy_sharpe"
    ),
    stability_penalty=1.0,
    min_local_parameter_count=1,
)

assert not smoke_result[
    "windows"
].empty

assert not smoke_result[
    "selected_parameters"
].empty

assert not smoke_result[
    "test_detail"
].empty

assert not smoke_result[
    "test_summary"
].empty

assert (
    smoke_result[
        "selected_parameters"
    ]["fold_id"].nunique()
    == len(
        smoke_result["windows"]
    )
)

print(
    "Walk-Forward 冒烟测试通过"
)

Walk-Forward 冒烟测试通过


## 正式测试

In [ ]:
walk_forward_result = (
    run_ma_walk_forward(
        stock_list=stock_list,
        parameter_grid=parameter_grid,
        train_months=36,
        test_months=12,
        step_months=12,
        window_type="rolling",
        start_date=None,
        end_date="2024-12-31",
        include_partial_test=False,
        commission_rate=0.0003,
        slippage_rate=0.0002,
        annual_risk_free_rate=0.0,
        trading_days=252,
        robustness_metric=(
            "avg_strategy_sharpe"
        ),
        stability_penalty=1.0,
        min_local_parameter_count=9,
    )
)

windows = walk_forward_result[
    "windows"
]

selected_parameters = (
    walk_forward_result[
        "selected_parameters"
    ]
)

parameter_frequency = (
    walk_forward_result[
        "parameter_frequency"
    ]
)

test_detail = walk_forward_result[
    "test_detail"
]

test_summary = walk_forward_result[
    "test_summary"
]

In [32]:
selection_columns = [
    "fold_id",
    "train_start",
    "train_end",
    "test_start",
    "test_end",
    "selected_ma_param",
    "train_avg_strategy_sharpe",
    "train_avg_excess_annual_return",
    "train_return_win_rate",
    "train_local_mean",
    "train_local_std",
    "train_local_min",
    "train_robustness_score",
]

display(
    selected_parameters[
        selection_columns
    ]
)

,fold_id,train_start,train_end,test_start,test_end,selected_ma_param,train_avg_strategy_sharpe,train_avg_excess_annual_return,train_return_win_rate,train_local_mean,train_local_std,train_local_min,train_robustness_score
0,1,2018-06-11,2020-06-10,2020-06-11,2021-06-10,30/60,0.6519,-0.1411,0.2000,0.6774,0.0525,0.5738,0.6249
1,2,2019-06-11,2021-06-10,2021-06-11,2022-06-10,35/110,1.2067,-0.1202,0.4000,1.1409,0.0438,1.0769,1.0971
2,3,2020-06-11,2022-06-10,2022-06-11,2023-06-10,25/110,0.6675,0.0754,0.8000,0.6262,0.0236,0.5938,0.6025
3,4,2021-06-11,2023-06-10,2023-06-11,2024-06-10,10/40,-0.3501,0.1072,1.0000,-0.3281,0.0983,-0.4800,-0.4263


In [33]:
display(parameter_frequency)

,selected_fast_window,selected_slow_window,selected_ma_param,selected_count,first_selected_fold,last_selected_fold,selected_rate
0,10,40,10/40,1,4,4,0.2500
1,25,110,25/110,1,3,3,0.2500
2,30,60,30/60,1,1,1,0.2500
3,35,110,35/110,1,2,2,0.2500


In [34]:
parameter_changes = (
    selected_parameters[
        [
            "fold_id",
            "selected_fast_window",
            "selected_slow_window",
            "selected_ma_param",
        ]
    ]
    .copy()
)

parameter_changes[
    "fast_change"
] = (
    parameter_changes[
        "selected_fast_window"
    ]
    .diff()
    .abs()
)

parameter_changes[
    "slow_change"
] = (
    parameter_changes[
        "selected_slow_window"
    ]
    .diff()
    .abs()
)

parameter_changes[
    "distance_to_10_40"
] = (
    (
        parameter_changes[
            "selected_fast_window"
        ]
        - 10
    ).abs()
    + (
        parameter_changes[
            "selected_slow_window"
        ]
        - 40
    ).abs()
)

display(parameter_changes)

,fold_id,selected_fast_window,selected_slow_window,selected_ma_param,fast_change,slow_change,distance_to_10_40
0,1,30,60,30/60,NaN,NaN,40
1,2,35,110,35/110,5.0000,50.0000,95
2,3,25,110,25/110,10.0000,0.0000,85
3,4,10,40,10/40,15.0000,70.0000,0


In [35]:
train_test_comparison = selected_parameters.merge(
    test_summary[
        [
            "fold_id",
            "selected_ma_param",
            "avg_strategy_sharpe",
            "avg_excess_annual_return",
            "return_win_rate",
            "avg_drawdown_improvement",
            "avg_trade_count",
        ]
    ],
    on=[
        "fold_id",
        "selected_ma_param",
    ],
    how="left",
)

train_test_comparison = (
    train_test_comparison.rename(
        columns={
            "avg_strategy_sharpe":
                "test_avg_strategy_sharpe",
            "avg_excess_annual_return":
                "test_avg_excess_annual_return",
            "return_win_rate":
                "test_return_win_rate",
            "avg_drawdown_improvement":
                "test_avg_drawdown_improvement",
            "avg_trade_count":
                "test_avg_trade_count",
        }
    )
)

train_test_comparison[
    "sharpe_change"
] = (
    train_test_comparison[
        "test_avg_strategy_sharpe"
    ]
    - train_test_comparison[
        "train_avg_strategy_sharpe"
    ]
)

train_test_comparison[
    "excess_return_change"
] = (
    train_test_comparison[
        "test_avg_excess_annual_return"
    ]
    - train_test_comparison[
        "train_avg_excess_annual_return"
    ]
)

display(
    train_test_comparison[
        [
            "fold_id",
            "selected_ma_param",
            "train_avg_strategy_sharpe",
            "test_avg_strategy_sharpe",
            "sharpe_change",
            "train_avg_excess_annual_return",
            "test_avg_excess_annual_return",
            "excess_return_change",
            "test_return_win_rate",
            "test_avg_drawdown_improvement",
        ]
    ]
)

,fold_id,selected_ma_param,train_avg_strategy_sharpe,test_avg_strategy_sharpe,sharpe_change,train_avg_excess_annual_return,test_avg_excess_annual_return,excess_return_change,test_return_win_rate,test_avg_drawdown_improvement
0,1,30/60,0.6519,1.2058,0.5539,-0.1411,-0.2626,-0.1215,0.0000,0.0254
1,2,35/110,1.2067,-1.0834,-2.2901,-0.1202,0.0980,0.2182,0.8000,0.1513
2,3,25/110,0.6675,-1.1628,-1.8303,0.0754,-0.0538,-0.1292,0.6000,0.0646
3,4,10/40,-0.3501,-0.3324,0.0177,0.1072,0.0788,-0.0284,0.6000,0.1454


In [36]:
fixed_result = run_fixed_parameter_validation(
    stock_list=stock_list,
    parameters=[
        (10, 40),
        (20, 60),
        (25, 110),
    ],
    windows=windows,
    commission_rate=0.0003,
    slippage_rate=0.0002,
    annual_risk_free_rate=0.0,
    trading_days=252,
)

fixed_summary = fixed_result["summary"]

fixed_overall = (
    fixed_summary
    .groupby(
        "ma_param",
        as_index=False,
    )
    .agg(
        fold_count=(
            "fold_id",
            "nunique",
        ),
        avg_test_strategy_sharpe=(
            "avg_strategy_sharpe",
            "mean",
        ),
        median_test_strategy_sharpe=(
            "avg_strategy_sharpe",
            "median",
        ),
        avg_test_excess_return=(
            "avg_excess_annual_return",
            "mean",
        ),
        median_test_excess_return=(
            "avg_excess_annual_return",
            "median",
        ),
        positive_excess_fold_rate=(
            "avg_excess_annual_return",
            lambda values: (
                values > 0
            ).mean(),
        ),
        avg_return_win_rate=(
            "return_win_rate",
            "mean",
        ),
        avg_drawdown_improvement=(
            "avg_drawdown_improvement",
            "mean",
        ),
        avg_trade_count=(
            "avg_trade_count",
            "mean",
        ),
    )
)

display(
    fixed_overall.sort_values(
        "avg_test_strategy_sharpe",
        ascending=False,
    )
)

,ma_param,fold_count,avg_test_strategy_sharpe,median_test_strategy_sharpe,avg_test_excess_return,median_test_excess_return,positive_excess_fold_rate,avg_return_win_rate,avg_drawdown_improvement,avg_trade_count
1,20/60,4,-0.0434,-0.3023,-0.0169,0.0586,0.7500,0.5500,0.1265,4.4000
0,10/40,4,-0.1445,-0.1252,-0.0602,0.0966,0.7500,0.6000,0.1090,7.1500
2,25/110,4,-0.1992,-0.5487,0.0122,0.0020,0.5000,0.6000,0.1196,2.5000


In [37]:
dynamic_overall = pd.DataFrame(
    {
        "ma_param": [
            "dynamic_walk_forward"
        ],
        "fold_count": [
            test_summary[
                "fold_id"
            ].nunique()
        ],
        "avg_test_strategy_sharpe": [
            test_summary[
                "avg_strategy_sharpe"
            ].mean()
        ],
        "median_test_strategy_sharpe": [
            test_summary[
                "avg_strategy_sharpe"
            ].median()
        ],
        "avg_test_excess_return": [
            test_summary[
                "avg_excess_annual_return"
            ].mean()
        ],
        "median_test_excess_return": [
            test_summary[
                "avg_excess_annual_return"
            ].median()
        ],
        "positive_excess_fold_rate": [
            (
                test_summary[
                    "avg_excess_annual_return"
                ] > 0
            ).mean()
        ],
        "avg_return_win_rate": [
            test_summary[
                "return_win_rate"
            ].mean()
        ],
        "avg_drawdown_improvement": [
            test_summary[
                "avg_drawdown_improvement"
            ].mean()
        ],
        "avg_trade_count": [
            test_summary[
                "avg_trade_count"
            ].mean()
        ],
    }
)

overall_comparison = pd.concat(
    [
        fixed_overall,
        dynamic_overall,
    ],
    ignore_index=True,
)

display(
    overall_comparison.sort_values(
        "avg_test_strategy_sharpe",
        ascending=False,
    )
)

,ma_param,fold_count,avg_test_strategy_sharpe,median_test_strategy_sharpe,avg_test_excess_return,median_test_excess_return,positive_excess_fold_rate,avg_return_win_rate,avg_drawdown_improvement,avg_trade_count
1,20/60,4,-0.0434,-0.3023,-0.0169,0.0586,0.7500,0.5500,0.1265,4.4000
0,10/40,4,-0.1445,-0.1252,-0.0602,0.0966,0.7500,0.6000,0.1090,7.1500
2,25/110,4,-0.1992,-0.5487,0.0122,0.0020,0.5000,0.6000,0.1196,2.5000
3,dynamic_walk_forward,4,-0.3432,-0.7079,-0.0349,0.0125,0.5000,0.5000,0.0967,4.0000
